# Healthcare Disease Prediction – Stage 3 and Stage 4
**Course:** CDS4003 – Data Science for Business Application

This notebook performs:
- **Stage 3:** Data understanding, quality checks, cleaning and documentation.
- **Stage 4:** Exploratory Data Analysis (EDA), visualisations and feature engineering.

**Dataset:** `Healthcare Disease Prediction.xlsx`  
**Target column:** `label` (Normal / Abnormal)

Run the notebook from top to bottom. Read the Markdown notes and keep the printed results and charts as evidence for your project.

## 0. Choose where to run this notebook

**Recommended: Google Colab** if you want the easiest setup and do not want to install Python libraries. Open https://colab.research.google.com, upload this notebook, then upload your Excel dataset when the upload window appears in the next cell.

You can also use Jupyter Notebook if it is already installed. In that case, put the Excel file in the same folder as this notebook.

The next cell works in either environment.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

file_name = "Healthcare Disease Prediction.xlsx"

if not os.path.exists(file_name):
    try:
        from google.colab import files
        uploaded = files.upload()
        # Use the first uploaded Excel file
        file_name = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError(
            "Put 'Healthcare Disease Prediction.xlsx' in the same folder "
            "as this notebook, then run this cell again."
        )

df = pd.read_excel(file_name)

print("Dataset loaded successfully.")
print("Original dataset shape:", df.shape)
display(df.head())

# Stage 3 – Data Understanding and Cleaning

## 3.1 Understand the dataset

Check the number of rows and columns, column names, data types and summary statistics.

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nNumerical summary:")
display(df.describe().T)

print("\nCategorical summary:")
display(df.describe(include="object").T)

## 3.2 Check missing values

The code counts missing values in every column and calculates the total. Missing numerical values, if any, are filled with the median; missing categorical values are filled with the mode. The median is less affected by extreme values than the mean, while the mode preserves the most common category.

If the dataset has no missing values, no imputation is performed.

In [ ]:
missing_before = df.isna().sum().sort_values(ascending=False)
print("Missing values by column:")
display(missing_before[missing_before > 0])

print("Total missing values before cleaning:", int(df.isna().sum().sum()))

# Fill missing numerical values with the median
numeric_cols_before = df.select_dtypes(include=np.number).columns
for col in numeric_cols_before:
    if df[col].isna().any():
        df[col] = df[col].fillna(df[col].median())

# Fill missing categorical/text values with the most frequent value
categorical_cols_before = df.select_dtypes(include=["object", "category"]).columns
for col in categorical_cols_before:
    if df[col].isna().any():
        mode_values = df[col].mode(dropna=True)
        if not mode_values.empty:
            df[col] = df[col].fillna(mode_values.iloc[0])

print("Total missing values after treatment:", int(df.isna().sum().sum()))

## 3.3 Check and remove duplicate records

Duplicate rows can give repeated observations extra influence in the analysis. Count them first, remove exact duplicate rows, and compare the dataset size before and after.

In [ ]:
rows_before_duplicates = len(df)
duplicate_count = int(df.duplicated().sum())

print("Duplicate rows found:", duplicate_count)

df = df.drop_duplicates().copy()

print("Rows before duplicate removal:", rows_before_duplicates)
print("Rows after duplicate removal:", len(df))
print("Rows removed:", rows_before_duplicates - len(df))
print("Remaining duplicate rows:", int(df.duplicated().sum()))

## 3.4 Check inconsistent values and formats

Text columns are checked for leading/trailing spaces and category spellings. The code removes extra spaces and prints the unique values so unusual labels can be reviewed. It does not automatically rename legitimate categories.

In [ ]:
text_cols = df.select_dtypes(include=["object", "category"]).columns

for col in text_cols:
    # Remove leading/trailing spaces from text values
    df[col] = df[col].astype("string").str.strip()

print("Unique values in categorical columns:")
for col in text_cols:
    print(f"\n{col}:")
    print(sorted(df[col].dropna().unique().tolist(), key=str))

print("\nData types after checking:")
display(df.dtypes.astype(str).to_frame("Data type"))

## 3.5 Check unusual or impossible values

First inspect the minimum and maximum of numerical columns. The checks below flag values that may be invalid for common health measurements. They are review flags, not automatic deletion rules. Confirm units and dataset documentation before changing any values.

In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns
display(df[numeric_cols].agg(["min", "max"]).T)

# Basic plausibility checks for selected columns, if present
plausibility_rules = {
    "age": lambda s: s < 0,
    "bmi": lambda s: s <= 0,
    "sleep_hours": lambda s: (s < 0) | (s > 24),
    "systolic_bp": lambda s: s <= 0,
    "diastolic_bp": lambda s: s <= 0,
    "heart_rate": lambda s: s <= 0,
    "hdl": lambda s: s <= 0,
}

print("Potentially invalid values to review:")
for col, rule in plausibility_rules.items():
    if col in df.columns:
        count = int(rule(df[col]).sum())
        print(f"{col}: {count}")

## 3.6 Detect potential outliers using the IQR method

The Interquartile Range (IQR) method identifies values below Q1 − 1.5 × IQR or above Q3 + 1.5 × IQR. Medical measurements can be naturally extreme and may be important for prediction, so this notebook reports potential outliers but does not delete them automatically.

In [ ]:
# Exclude binary indicator columns from the outlier summary
binary_cols = [
    col for col in numeric_cols
    if df[col].dropna().nunique() <= 2
]

outlier_rows = []
for col in numeric_cols:
    if col in binary_cols:
        continue

    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    count = int(((df[col] < lower) | (df[col] > upper)).sum())

    outlier_rows.append({
        "Column": col,
        "Lower IQR limit": lower,
        "Upper IQR limit": upper,
        "Potential outliers": count,
        "Percentage": count / len(df) * 100
    })

outlier_report = pd.DataFrame(outlier_rows).sort_values(
    "Potential outliers", ascending=False
)
display(outlier_report)

## 3.7 Final data quality check and save the cleaned dataset

Compare the main quality indicators after cleaning. The cleaned file is saved separately so the original Excel file remains unchanged.

In [ ]:
print("FINAL CLEANING SUMMARY")
print("----------------------")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", int(df.isna().sum().sum()))
print("Duplicate rows:", int(df.duplicated().sum()))

cleaned_file = "Healthcare_Cleaned.xlsx"
df.to_excel(cleaned_file, index=False)
print("\nSaved:", cleaned_file)

# Stage 4 – Exploratory Data Analysis (EDA) and Feature Engineering

## 4.1 Descriptive statistics

Review numerical summaries and the frequency of categories. These results help describe the dataset before modelling.

In [ ]:
print("Numerical descriptive statistics:")
display(df.describe().T)

print("\nTarget variable counts:")
display(df["label"].value_counts(dropna=False).to_frame("Count"))

print("\nTarget variable percentages:")
display(
    (df["label"].value_counts(normalize=True, dropna=False) * 100)
    .round(2).to_frame("Percentage")
)

## 4.2 Visualisation 1 – Distribution of the target variable

This bar chart compares the number of Normal and Abnormal records. If one class is much larger, the modelling stage should consider class imbalance and evaluate metrics such as recall, precision and F1-score in addition to accuracy.

In [ ]:
plt.figure(figsize=(7, 5))
sns.countplot(data=df, x="label", order=df["label"].value_counts().index)
plt.title("Distribution of Health Classification")
plt.xlabel("Health classification")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 4.3 Compare health measurements by target class

The code calculates the mean of available health measurements for Normal and Abnormal records. It checks that the expected columns exist before running.

In [ ]:
health_columns = [
    "age", "bmi", "glucose", "HbA1c_level",
    "cholesterol", "sleep_hours"
]
health_columns = [col for col in health_columns if col in df.columns]

health_comparison = df.groupby("label")[health_columns].mean().round(2)
display(health_comparison)

## 4.4 Visualisation 2 – BMI and glucose

The scatter plot explores whether BMI and glucose values show different patterns across the two health classifications. A pattern or overlap does not prove that either measurement causes the classification.

In [ ]:
if {"bmi", "glucose", "label"}.issubset(df.columns):
    plt.figure(figsize=(9, 6))
    sns.scatterplot(
        data=df, x="bmi", y="glucose",
        hue="label", alpha=0.35, s=25
    )
    plt.title("BMI and Glucose by Health Classification")
    plt.xlabel("BMI")
    plt.ylabel("Glucose")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns for this chart were not found.")

## 4.5 Visualisation 3 – Age distribution by health classification

A box plot compares the median, spread and potential extreme age values in the two groups.

In [ ]:
if {"age", "label"}.issubset(df.columns):
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=df, x="label", y="age")
    plt.title("Age Distribution by Health Classification")
    plt.xlabel("Health classification")
    plt.ylabel("Age")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns for this chart were not found.")

## 4.6 Lifestyle factors and health classification

The tables show the percentage of Normal and Abnormal records within each lifestyle category. This is more useful for comparison than raw counts when category sizes differ.

In [ ]:
lifestyle_columns = [
    "smoking", "family_history",
    "high_blood_pressure", "physical_activity"
]

for col in lifestyle_columns:
    if col in df.columns:
        print(f"\nHealth classification by {col} (row percentages):")
        table = pd.crosstab(
            df[col], df["label"], normalize="index"
        ) * 100
        display(table.round(2))

## 4.7 Visualisation 4 – Smoking status and health classification

The chart compares the percentage of each health classification within every smoking category.

In [ ]:
if {"smoking", "label"}.issubset(df.columns):
    smoking_table = pd.crosstab(
        df["smoking"], df["label"], normalize="index"
    ) * 100

    ax = smoking_table.plot(
        kind="bar", figsize=(8, 5), rot=0
    )
    ax.set_title("Smoking Status and Health Classification")
    ax.set_xlabel("Smoking status")
    ax.set_ylabel("Percentage within smoking category (%)")
    ax.legend(title="Health classification")
    plt.tight_layout()
    plt.show()
else:
    print("Required columns for this chart were not found.")

## 4.8 Visualisation 5 – Correlation between numerical variables

A correlation heatmap helps identify numerical variables that move together. Correlation does not prove causation. Do not interpret correlations with engineered or target-derived fields as proof of predictive value.

In [ ]:
corr_cols = df.select_dtypes(include=np.number).columns.tolist()

if len(corr_cols) > 1:
    corr_matrix = df[corr_cols].corr()

    plt.figure(figsize=(14, 10))
    sns.heatmap(
        corr_matrix, cmap="coolwarm", center=0,
        annot=False, linewidths=0.2
    )
    plt.title("Correlation Heatmap of Numerical Features")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numerical columns for a correlation heatmap.")

## 4.9 Feature Engineering

Create two additional numerical features:

1. **Pulse Pressure** = systolic blood pressure − diastolic blood pressure.
2. **Cholesterol Ratio** = total cholesterol ÷ HDL cholesterol.

The existing `age_level` and `bmi_level` columns can also be retained as age-group and BMI-category information; they are not newly calculated features.

The cholesterol ratio is set to missing when HDL is zero to avoid division by zero. Review the units and meaning of the source columns before using these features in a model.

In [ ]:
if {"systolic_bp", "diastolic_bp"}.issubset(df.columns):
    df["Pulse_Pressure"] = df["systolic_bp"] - df["diastolic_bp"]

if {"cholesterol", "hdl"}.issubset(df.columns):
    df["Cholesterol_Ratio"] = df["cholesterol"] / df["hdl"].replace(0, np.nan)

if "age_level" in df.columns:
    df["Age_Group"] = df["age_level"]

if "bmi_level" in df.columns:
    df["BMI_Category"] = df["bmi_level"]

new_feature_cols = [
    col for col in [
        "Pulse_Pressure", "Cholesterol_Ratio",
        "Age_Group", "BMI_Category"
    ] if col in df.columns
]

print("New/reused feature columns:")
display(df[new_feature_cols].head())

## 4.10 Evaluate the engineered numerical features

Compare the average Pulse Pressure and Cholesterol Ratio between the Normal and Abnormal groups. This is an initial descriptive check, not proof that the features improve model performance. Their value should later be tested using validation data.

In [ ]:
engineered_numeric = [
    col for col in ["Pulse_Pressure", "Cholesterol_Ratio"]
    if col in df.columns
]

if engineered_numeric:
    display(df.groupby("label")[engineered_numeric].mean().round(2))
else:
    print("No engineered numerical features were created.")

## 4.11 Important – Check possible target leakage before modelling

The dataset contains columns such as `disease_flags`, `sublabel`, `source_dataset`, and `composite_key`. These may encode information used to create the target `label`. `diabetes` may also be strongly related to the target.

Check their relationship with `label` before Stage 5. Do not automatically use these columns as model inputs. Decide whether each feature would genuinely be available at the time a real prediction is made. The final feature list should be justified and checked to avoid target leakage.

In [ ]:
possible_leakage_cols = [
    "diabetes", "disease_flags",
    "sublabel", "source_dataset", "composite_key"
]

for col in possible_leakage_cols:
    if col in df.columns:
        print(f"\n{col} versus label:")
        display(pd.crosstab(df[col], df["label"]))

## 4.12 Save the final EDA and feature-engineered dataset

This file includes the cleaned data and the engineered features. Keep the cleaned file and this final file separate.

In [ ]:
final_file = "Healthcare_Final_EDA_Features.xlsx"
df.to_excel(final_file, index=False)

print("Final EDA and feature-engineered dataset saved as:", final_file)
print("Final dataset shape:", df.shape)

## 4.13 Notes for the project report

Use your actual outputs when writing the report. Do not copy expected values from another run.

For Stage 3, report:
- Original and final row/column counts.
- Missing values and duplicate counts before and after cleaning.
- Categories and data types checked.
- IQR outlier counts and why you retained or treated the flagged values.

For Stage 4, report:
- Target class counts and percentages.
- The mean comparison table.
- What each chart shows in relation to the business problem.
- The two engineered numerical features and why they may be useful.
- Any target leakage risks identified.

Remember: descriptive relationships are associations, not proof of medical causation. This dataset is for an academic prediction project and should not be used to diagnose people.